In [1]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd



In [2]:
# Define the LaBSE model name
model_name = "sentence-transformers/LaBSE"

# Input and output file paths
input_file = "biobert_embedding_terms.csv"  # Replace with your input file path
output_file = "tumor_embeddings_labse.csv"  # Replace with your desired output file path



In [3]:
# Load tumor names from the CSV file
df = pd.read_csv(input_file)
if "Tumor_Names" not in df.columns:
    raise ValueError("The column 'Tumor_Names' is not found in the input file.")

tumor_names = df["Tumor_Names"].tolist()



In [4]:
# Load tokenizer and model
print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Set model to evaluation mode



Loading model: sentence-transformers/LaBSE


tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2025-01-23 15:24:38.752216: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-23 15:24:38.806437: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-23 15:24:38.806476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-23 15:24:38.807605: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-23 15:24:38.814399: I tensorflow/core/platform/cpu_feature_guar

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(501153, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

In [5]:
# Generate embeddings
embeddings = []
print("Generating embeddings...")
with torch.no_grad():
    for name in tumor_names:
        # Tokenize and process the input
        inputs = tokenizer(name, return_tensors="pt", padding=True, truncation=True)
        outputs = model(**inputs)
        # Use the CLS token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
        embeddings.append(cls_embedding)




Generating embeddings...


In [6]:
# Create a DataFrame for embeddings
embedding_columns = [f"dim_{i}" for i in range(len(embeddings[0]))]
embeddings_df = pd.DataFrame(embeddings, columns=embedding_columns)
embeddings_df["Tumor_Names"] = tumor_names

# Save to CSV
embeddings_df.to_csv(output_file, index=False)
print(f"Embeddings saved to {output_file}")

Embeddings saved to tumor_embeddings_labse.csv
